# Level 1 — Scientific Problem Framing and Python Foundation

## Problem Statement: HydroSense-Kenya Smart Irrigation System
In the Arid and Semi-Arid Lands (ASALs) of Kenya, agricultural productivity is fundamentally constrained by water availability and the increasing unpredictability of climate patterns. Smallholder farmers, agricultural innovation hubs, and university demonstration farms face severe, daily trade-offs between water conservation, crop yield, and the energy costs associated with pumping groundwater or surface reserves. Traditional irrigation scheduling in these regions often relies on calendar-based approaches or subjective visual inspection of the topsoil. These unscientific methods inevitably lead to one of two detrimental outcomes: critical crop water stress (under-irrigation) which permanently stunts crop development, or wasted water and excessive nutrient leaching (over-irrigation) which unnecessarily drives up the electrical costs of running water pumps.

The HydroSense-Kenya project aims to address this critical agricultural challenge by developing a computational, climate-aware decision-support system using Python. By transitioning from guesswork to a data-driven mathematical model, the system tracks the physical movement of water into and out of the root zone. The central scientific question we are answering is: *Given specific local weather parameters (temperature, humidity, solar radiation, wind speed) and discrete soil-sensor readings, how can we mathematically model water availability, accurately simulate future soil moisture depletion, and recommend an optimized irrigation strategy that minimizes total water withdrawal without allowing moisture to drop below crop-specific wilting points?*

To achieve this, we rely on the fundamental discrete mass-balance equation of soil moisture:
$S_{t+1} = S_t + R_t + I_t - ET_t - D_t$

Where:
*   **$S_t$**: Soil water storage at time $t$.
*   **$R_t$**: Rainfall contribution.
*   **$I_t$**: Irrigation applied (our control variable).
*   **$ET_t$**: Evapotranspiration (crop water loss).
*   **$D_t$**: Drainage or water lost beyond the field capacity.

The primary water loss driver, Evapotranspiration ($ET$), is notoriously difficult to measure directly. Therefore, we estimate it using a simplified empirical model based on temperature, wind speed, humidity, and solar index. By translating this real-world physics problem into a vectorized Python simulation, we can utilize numerical methods (like root-finding and differential equations) to optimize the exact timing and volume of irrigation ($I_t$). Furthermore, by incorporating Monte Carlo simulations later in the project, we can account for the inherent uncertainty in Kenyan weather forecasts, allowing the farm to prepare for worst-case drought scenarios.

**Assumptions and Limitations of the Initial Model:**
1. **Homogeneous Soil Profile:** We assume the soil compartment is a single homogeneous layer, ignoring the complex multi-layer matric potential of real soil.
2. **Empirical ET Limitations:** The empirical ET formula provided is a localized approximation. It does not account for specific aerodynamic crop canopy resistance or stomatal behavior, which a more complex model like the Penman-Monteith equation would include.
3. **Negligible Runoff:** Surface runoff is assumed to be zero for this specific farm's topography; all rainfall is assumed to infiltrate the soil until field capacity is reached.
4. **Sensor Reliability:** We assume that once sensor faults are cleaned from the dataset, the remaining volumetric water content readings perfectly represent the entire crop zone.

## Data Dictionary

### 1. weather_daily.csv
* `date`: The calendar date of the reading.
* `rainfall_mm`: Total daily precipitation ($R_t$) in millimeters.
* `temperature_c`: Mean daily temperature in Celsius ($T$).
* `humidity_pct`: Relative humidity percentage.
* `wind_speed_mps`: Wind speed in meters per second ($W$).
* `solar_index`: Normalized solar radiation intensity ($Solar$).

### 2. soil_sensor_data.csv
* `timestamp`: Date and time of the noon reading.
* `zone_id`: The specific crop zone (Zone A, B, or C).
* `soil_moisture_pct`: Volumetric water content of the soil ($S_t$).
* `tank_level_liters`: Water remaining in the primary storage tank.
* `pump_flow_lpm`: The flow rate of the irrigation pump in Liters Per Minute.
* `pump_power_watts`: The energy draw of the pump.
* `sensor_status`: Health of the hardware ('OK' or fault codes).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# 1. Load Initial Datasets
repo_root = Path('.').resolve().parent

# Load the data normally (since we fixed the CSVs to use commas)
weather = pd.read_csv(repo_root / 'data' / 'raw' / 'weather_daily.csv', na_values=["NA", ""])

# Ensure date is parsed correctly
weather['date'] = pd.to_datetime(weather['date'])

print(f"Loaded {len(weather)} days of weather data successfully.\n")

# Inspect the structure as required by the brief
print("--- Weather Data Structure ---")
weather.info()

In [ ]:
# 2. Reusable Python Functions for Core Domain

def calc_empirical_et(temp, wind, solar, humidity):
    """
    Computes simplified daily Evapotranspiration (ET) in mm.
    Formula defined in project brief: ET = max(0, 0.12*T + 0.35*W + 2.4*Solar - 0.025*H)
    """
    et = 0.12 * temp + 0.35 * wind + 2.4 * solar - 0.025 * humidity
    return max(0.0, et)

def calc_water_balance(s_current, rain, et, drainage, irrigation):
    """
    Computes next day's soil moisture based on the discrete mass balance equation.
    S_(t+1) = S_t + R_t + I_t - ET_t - D_t
    """
    s_next = s_current + rain + irrigation - et - drainage
    return max(0.0, s_next)

print("Core mathematical functions defined successfully.")

In [ ]:
# 3. Basic Scientific Plot (Baseline Visualization)
plt.figure(figsize=(10, 4))
plt.bar(weather['date'], weather['rainfall_mm'], color='royalblue')
plt.title("Daily Rainfall (mm) - March 2026")
plt.xlabel("Date")
plt.ylabel("Rainfall (mm)")
plt.grid(axis='y', alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()